# KOTRA 해외바이어 크롤러

이 노트북은 KOTRA 해외바이어 정보를 크롤링하여 Excel 및 텍스트 파일로 저장합니다.

## 사용 방법
1. 아래 셀들을 순서대로 실행하세요 (Shift + Enter)
2. 크롤링이 완료되면 자동으로 파일이 다운로드됩니다

## 수집 정보
- **대상 국가**: 22개국
- **HS CODE**: 391810
- **데이터**: 수입기업명, 거래국가수, 거래건수, 총거래금액, 수입예측값, 한국수입여부

## 1️⃣ 환경 설정 (약 1-2분 소요)

In [ ]:
%%bash
# Chrome 및 ChromeDriver 설치
apt-get update
apt-get install -y chromium-chromedriver
cp /usr/lib/chromium-browser/chromedriver /usr/bin

# 파이썬 패키지 설치
pip install -q selenium pandas openpyxl

## 2️⃣ 크롤러 코드 로드

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import Select
import pandas as pd
import time
from typing import List, Dict
import re


class KotraSeleniumCrawler:
    def __init__(self, headless: bool = True):
        chrome_options = Options()
        if headless:
            chrome_options.add_argument('--headless')
            chrome_options.add_argument('--no-sandbox')
            chrome_options.add_argument('--disable-dev-shm-usage')
        chrome_options.add_argument('--disable-blink-features=AutomationControlled')
        chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
        
        self.driver = webdriver.Chrome(options=chrome_options)
        self.wait = WebDriverWait(self.driver, 20)
        self.base_url = "https://www.kotra.or.kr/bigdata/partner/search"

    def setup_page(self, country_code: str, hs_code: str):
        try:
            self.driver.get(self.base_url)
            time.sleep(2)
            
            country_select = self.wait.until(EC.presence_of_element_located((By.ID, "country-list-ex")))
            select = Select(country_select)
            select.select_by_value(country_code)
            time.sleep(1)
            
            hs_input = self.driver.find_element(By.ID, "hscode-input-ex")
            hs_input.clear()
            hs_input.send_keys(hs_code)
            time.sleep(1)
            
            search_button = self.driver.find_element(By.CSS_SELECTOR, "button.btn-search")
            search_button.click()
            time.sleep(3)
            
            return True
        except Exception as e:
            print(f"페이지 설정 중 오류: {str(e)}")
            return False

    def extract_table_data(self) -> List[Dict]:
        data = []
        try:
            rows = self.driver.find_elements(By.CSS_SELECTOR, "div[role='row'][row-index]")
            
            for row in rows:
                try:
                    cells = row.find_elements(By.CSS_SELECTOR, "div[role='gridcell']")
                    
                    if len(cells) >= 7:
                        company_name = cells[1].text.strip()
                        
                        def parse_number(text):
                            cleaned = re.sub(r'[,\s]', '', text)
                            try:
                                return int(cleaned)
                            except:
                                return 0
                        
                        row_data = {
                            '수입기업': company_name,
                            '거래국가수': parse_number(cells[2].text),
                            '거래건수': parse_number(cells[3].text),
                            '총거래금액(USD)': parse_number(cells[4].text),
                            '수입예측값': parse_number(cells[5].text),
                            '한국수입여부': cells[6].text.strip()
                        }
                        data.append(row_data)
                except:
                    continue
            
            return data
        except Exception as e:
            print(f"테이블 데이터 추출 오류: {str(e)}")
            return []

    def go_to_next_page(self) -> bool:
        try:
            next_button = self.driver.find_element(By.CSS_SELECTOR, "button[aria-label='Next Page']")
            if 'disabled' in next_button.get_attribute('class'):
                return False
            next_button.click()
            time.sleep(2)
            return True
        except:
            return False

    def crawl_country(self, country_code: str, country_name: str, hs_code: str) -> List[Dict]:
        print(f"\n{'='*60}")
        print(f"크롤링 시작: {country_name} ({country_code})")
        print(f"{'='*60}")
        
        all_data = []
        
        if not self.setup_page(country_code, hs_code):
            print(f"{country_name} 크롤링 실패")
            return []
        
        page = 1
        while True:
            print(f"  페이지 {page} 처리 중...")
            
            page_data = self.extract_table_data()
            
            if not page_data:
                print(f"  데이터가 없습니다.")
                break
            
            for item in page_data:
                item['수입국가'] = country_name
                item['수입국가코드'] = country_code
            
            all_data.extend(page_data)
            print(f"  {len(page_data)}개 데이터 수집 완료")
            
            if not self.go_to_next_page():
                print(f"  전체 {len(all_data)}건 수집 완료")
                break
            
            page += 1
            time.sleep(1)
        
        return all_data

    def crawl_all_countries(self, countries: List[Dict], hs_code: str) -> pd.DataFrame:
        all_data = []
        
        for i, country in enumerate(countries, 1):
            print(f"\n진행률: {i}/{len(countries)}")
            
            country_data = self.crawl_country(country['code'], country['name'], hs_code)
            all_data.extend(country_data)
            
            print(f"{country['name']} 완료: {len(country_data)}건")
            time.sleep(2)
        
        df = pd.DataFrame(all_data)
        
        if not df.empty:
            columns_order = [
                '수입국가', '수입국가코드', '수입기업', '거래국가수',
                '거래건수', '총거래금액(USD)', '수입예측값', '한국수입여부'
            ]
            df = df[columns_order]
        
        return df

    def close(self):
        if self.driver:
            self.driver.quit()

print("✅ 크롤러 클래스 로드 완료")

## 3️⃣ 크롤링 설정

In [ ]:
# 크롤링할 국가 목록
countries = [
    {'code': 'RU', 'name': '러시아연방'},
    {'code': 'MX', 'name': '멕시코'},
    {'code': 'US', 'name': '미국'},
    {'code': 'BD', 'name': '방글라데시'},
    {'code': 'VN', 'name': '베트남'},
    {'code': 'AR', 'name': '아르헨티나'},
    {'code': 'EC', 'name': '에콰도르'},
    {'code': 'UG', 'name': '우간다'},
    {'code': 'UZ', 'name': '우즈베키스탄'},
    {'code': 'IN', 'name': '인도'},
    {'code': 'ID', 'name': '인도네시아'},
    {'code': 'JP', 'name': '일본'},
    {'code': 'CL', 'name': '칠레'},
    {'code': 'KZ', 'name': '카자흐스탄'},
    {'code': 'KE', 'name': '케냐'},
    {'code': 'CO', 'name': '콜롬비아'},
    {'code': 'TR', 'name': '튀르키예'},
    {'code': 'PA', 'name': '파나마'},
    {'code': 'PY', 'name': '파라과이'},
    {'code': 'PK', 'name': '파키스탄'},
    {'code': 'PE', 'name': '페루'},
    {'code': 'PH', 'name': '필리핀'}
]

# HS CODE (필요시 변경)
hs_code = '391810'

print(f"✅ 설정 완료")
print(f"   - 대상 국가: {len(countries)}개국")
print(f"   - HS CODE: {hs_code}")

## 4️⃣ 크롤링 실행 (시간이 오래 걸릴 수 있습니다)

⚠️ 주의: 22개국 전체를 크롤링하면 30분 이상 소요될 수 있습니다.

In [ ]:
# 크롤러 초기화
crawler = KotraSeleniumCrawler(headless=True)

try:
    print("="*60)
    print("KOTRA 해외바이어 크롤링 시작")
    print(f"HS CODE: {hs_code}")
    print(f"대상 국가: {len(countries)}개국")
    print("="*60)
    
    # 크롤링 실행
    df = crawler.crawl_all_countries(countries, hs_code)
    
    print(f"\n\n✅ 크롤링 완료!")
    print(f"   총 수집 데이터: {len(df)}건")
    
finally:
    crawler.close()

## 5️⃣ 결과 확인

In [ ]:
# 데이터 미리보기
print("=== 데이터 미리보기 ===")
display(df.head(10))

print("\n=== 국가별 통계 ===")
country_stats = df.groupby('수입국가').size().reset_index(name='기업수')
display(country_stats)

print(f"\n=== 전체 통계 ===")
print(f"총 기업 수: {len(df)}개")
print(f"총 거래금액: ${df['총거래금액(USD)'].sum():,}")
print(f"한국 수입 기업: {len(df[df['한국수입여부'] == 'Y'])}개")

## 6️⃣ 파일 저장 및 다운로드

In [ ]:
from google.colab import files

# 엑셀 파일 저장
excel_filename = f'kotra_buyers_{hs_code}.xlsx'
df.to_excel(excel_filename, index=False, engine='openpyxl')
print(f"✅ 엑셀 파일 저장 완료: {excel_filename}")

# 탭 구분 텍스트 파일 저장
txt_filename = f'kotra_buyers_{hs_code}.txt'
df.to_csv(txt_filename, sep='\t', index=False, encoding='utf-8-sig')
print(f"✅ TXT 파일 저장 완료: {txt_filename}")

# 파일 다운로드
print("\n📥 파일 다운로드 중...")
files.download(excel_filename)
files.download(txt_filename)

print("\n🎉 모든 작업이 완료되었습니다!")

## 💡 팁

### 특정 국가만 크롤링하려면?
셀 3에서 `countries` 리스트를 수정하세요:
```python
countries = [
    {'code': 'US', 'name': '미국'},
    {'code': 'VN', 'name': '베트남'},
]
```

### 다른 HS CODE를 사용하려면?
셀 3에서 `hs_code` 변수를 수정하세요:
```python
hs_code = '123456'  # 원하는 HS CODE
```

### 크롤링 속도를 조절하려면?
셀 2의 `time.sleep()` 값을 조정하세요 (숫자가 클수록 느림)